# Can cache risk decide which requests need more xKV rank?

## tl;dr

The previous request router switched between three rank-64 allocation profiles.
It failed because the global answer-Fisher profile was already essentially exact.
This experiment keeps that profile and routes **rank** instead.

- Reuse the frozen answer-Fisher utility curves.
- Hash-split all 1,319 pinned GSM-Hard questions into 600 router-fit, 400 screen,
  and 319 locked-final examples.
- Predict rank-48/64/80 first-token KL from question features and spectral summaries
  of the dense pre-answer cache.
- Select request ranks under an aggregate modeled-bit budget exactly equal to fixed rank 64.
  Unused bits count as explicit policy padding.
- Open the final split only if the screen shows positive KL improvement, at least
  95% first-token fidelity, generation noninferiority, exact storage matching,
  and use of at least two ranks.

GSM-Hard is an external xKV-method holdout, not a pristine model benchmark:
dense CODI results on it were previously reported. This remains a modeled-storage
quality experiment rather than a native latency benchmark.

## Context & Methods

At the end of CODI's sixth latent step, every request has a dense K/V cache but
has not generated an answer token. The router measures spectral tail energy by
layer band and K/V tensor, combines those diagnostics with simple prompt features,
and predicts the relative dense first-token KL of ranks 48, 64, and 80.

The rank policy solves a constrained allocation problem across the evaluation
split. Requests predicted to tolerate compression can receive rank 48; saved bits
can fund rank 80 on fragile requests. The policy may leave bits unused, but those
bits are still charged as padding. Therefore its accounted storage is exactly the
fixed-rank-64 comparator budget.

### Key Assumptions

- Cache features are measured before the answer cue and contain no gold answer or
  generated answer token.
- The fit, screen, and final question hashes are disjoint and deterministic.
- The batch size is frozen at eight because padded cache length enters modeled
  storage accounting.
- A passing result supports quality at equal modeled bits, not production latency.

## 1. Reproducible setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "775ad2b3787617840d3e851223f92ce98842af31"
REPO_DIR = "/kaggle/working/latent-reasoning"
OUTPUT_DIR = "/kaggle/working/codi_cache_risk_rank_router"

REPRODUCTION_SUMMARY_INPUT = (
    "/kaggle/input/datasets/jonraza15/"
    "corrected-official-codi-answer-cue-endpoint-tsv-c/"
    "latent-reasoning/outputs/official_codi_gpt2/eval/"
    "revision_fd641b3d/full_gsm8k/summary.json"
)
PREVIOUS_ARTIFACT_INPUT = ""

FIT_EXAMPLES = 600
SCREEN_EXAMPLES = 400
FINAL_EXAMPLES = 319
BASELINE_RANK = 64

import glob, json, os, pathlib, subprocess, sys
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
assert len(RUN_COMMIT) == 40
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("code commit", subprocess.run(
    ["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip())

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.52.4", "datasets==3.6.0", "peft==0.15.2",
    "accelerate==1.7.0", "huggingface-hub>=0.34,<1.0",
    "safetensors==0.5.3", "pytest>=8,<10",
], check=True)
probe = subprocess.run([
    sys.executable, "-c",
    "from peft.import_utils import is_torchao_available\n"
    "try:\n print(is_torchao_available())\n"
    "except ImportError:\n print('incompatible')",
], capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_cache_risk_rank_router.py",
    "tests/test_cache_risk_rank_router_runner.py",
    "tests/test_adaptive_kv_allocation.py",
    "tests/test_request_adaptive_kv_router.py",
    "tests/test_official_codi_kv.py",
], check=True)

## 2. Locate and validate the frozen inputs

In [ ]:
import torch

def torch_contract(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False).get("contract")
    except Exception:
        return None

reproduction_suffix = "official_codi_gpt2/eval/revision_fd641b3d/full_gsm8k/summary.json"
reproduction_candidates = ([REPRODUCTION_SUMMARY_INPUT] if REPRODUCTION_SUMMARY_INPUT else []) + glob.glob(
    "/kaggle/input/**/" + reproduction_suffix, recursive=True
)
REPRODUCTION_SUMMARY = next((
    path for path in reproduction_candidates
    if pathlib.Path(path).is_file() and str(path).endswith(reproduction_suffix)
), None)
artifact_candidates = ([PREVIOUS_ARTIFACT_INPUT] if PREVIOUS_ARTIFACT_INPUT else []) + glob.glob(
    "/kaggle/input/**/adaptive_kv_allocation.pt", recursive=True
)
PREVIOUS_ARTIFACT = next((
    path for path in artifact_candidates
    if pathlib.Path(path).is_file() and torch_contract(path) == "official_codi_adaptive_kv_allocation_holdout_v1"
), None)
assert REPRODUCTION_SUMMARY, "Attach the completed official CODI reproduction dataset"
assert PREVIOUS_ARTIFACT, "Attach adaptive_kv_allocation.pt from the completed adaptive allocator"
previous = torch.load(PREVIOUS_ARTIFACT, map_location="cpu", weights_only=False)
assert previous["selected_candidate"]["name"] == "adaptive_kv_r64_w1"
assert previous["final_replication"]["gate"]["passed"] is False
print("reproduction", REPRODUCTION_SUMMARY)
print("adaptive predecessor", PREVIOUS_ARTIFACT)

## 3. Run the frozen cache-risk rank router

In [ ]:
command = [
    sys.executable, "-u", "scripts/run_codi_cache_risk_rank_router.py",
    "--config", "configs/official_codi_gpt2.yaml",
    "--reproduction-summary", REPRODUCTION_SUMMARY,
    "--previous-artifact", PREVIOUS_ARTIFACT,
    "--output-dir", OUTPUT_DIR,
    "--fit-examples", str(FIT_EXAMPLES),
    "--screen-examples", str(SCREEN_EXAMPLES),
    "--final-examples", str(FINAL_EXAMPLES),
    "--baseline-rank", str(BASELINE_RANK),
    "--ridge", "1.0", "--bootstrap-samples", "10000",
    "--minimum-first-token-fidelity", "0.95",
    "--accuracy-noninferiority-margin", "0.02",
    "--batch-size", "8", "--generation-batch-size", "8",
    "--max-new-tokens", "64", "--precision", "float32", "--device", "cuda",
]
subprocess.run(command, check=True)
summary = json.loads((pathlib.Path(OUTPUT_DIR) / "summary.json").read_text())
print(summary["decision"])

## 4. Split and storage-contract audit

In [ ]:
import pandas as pd
display(pd.DataFrame([{
    "dataset": summary["preregistration"]["dataset"],
    "fit": summary["preregistration"]["split_sizes"]["router_fit"],
    "screen": summary["preregistration"]["split_sizes"]["screen"],
    "locked final": summary["preregistration"]["split_sizes"]["final"],
    "candidate ranks": summary["preregistration"]["candidate_ranks"],
    "fixed budget rank": summary["preregistration"]["fixed_comparator_rank"],
}]))
display(pd.DataFrame([
    {"split": name, "question hash": value}
    for name, value in summary["split_hashes"].items()
]))

## 5. Router fit diagnostics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fit = summary["router_fit"]
display(pd.DataFrame([{
    **{f"mean KL rank {rank}": value for rank, value in fit["mean_kl_by_rank"].items()},
    "rank 48 routed": fit["budgeted_rank_counts"]["48"],
    "rank 64 routed": fit["budgeted_rank_counts"]["64"],
    "rank 80 routed": fit["budgeted_rank_counts"]["80"],
    "policy padding bits": fit["budgeted_router"]["policy_budget_padding_bits"],
}]))

coefficients = np.asarray(summary["router"]["coefficients"])[1:, :]
importance = np.linalg.norm(coefficients, axis=1)
top = np.argsort(importance)[-15:]
fig, ax = plt.subplots(figsize=(9.5, 6.5), constrained_layout=True)
limit = max(1e-12, np.abs(coefficients[top]).max())
image = ax.imshow(coefficients[top], cmap="coolwarm", vmin=-limit, vmax=limit, aspect="auto")
ax.set_xticks(range(3), ["rank 48", "rank 64", "rank 80"])
ax.set_yticks(range(len(top)), np.asarray(summary["router"]["feature_names"])[top])
ax.set(title="Fifteen strongest frozen rank-risk coefficients",
       xlabel="predicted rank loss", ylabel="pre-answer feature")
fig.colorbar(image, ax=ax, label="standardized ridge coefficient")
plt.show()

## 6. External screen

In [ ]:
screen = summary["screen"]
display(pd.DataFrame([
    {"arm": name, **record} for name, record in screen["generation"].items()
])[["arm", "accuracy", "correct", "examples", "exact_sequence_agreement"]])
display(pd.DataFrame([{
    "routed KL": screen["mean_routed_kl_from_dense"],
    "fixed rank-64 KL": screen["mean_fixed_rank64_kl_from_dense"],
    "mean KL gain": screen["mean_kl_gain_over_fixed_rank64"],
    "KL lower": screen["kl_gain_bootstrap_95ci"][0],
    "KL upper": screen["kl_gain_bootstrap_95ci"][1],
    "first-token fidelity": screen["first_token_fidelity"],
    "accounted storage ratio": screen["budget"]["accounted_storage_ratio"],
}]))
display(pd.DataFrame([screen["gate"]]))

counts = screen["rank_distribution"]
gain = screen["mean_kl_gain_over_fixed_rank64"]
lower, upper = screen["kl_gain_bootstrap_95ci"]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
axes[0].bar(counts.keys(), counts.values(), color="#315f8c")
axes[0].set(title="Ranks assigned on the 400-question screen",
            xlabel="request rank", ylabel="questions")
axes[1].errorbar([0], [gain], yerr=[[gain-lower], [upper-gain]], fmt="o", capsize=5,
                 color="#315f8c")
axes[1].axhline(0, color="#222222", linewidth=1)
axes[1].set_xticks([0], ["rank router − fixed r64"])
axes[1].set(title="Dense first-token KL gain with paired 95% interval",
            ylabel="fixed-r64 KL − routed KL (positive favors router)")
plt.show()

## 7. Locked final result

In [ ]:
if summary["final"] is None:
    print("The screen failed, so the locked 319-question final split was not evaluated.")
else:
    final = summary["final"]
    display(pd.DataFrame([{
        "mean KL gain": final["mean_kl_gain_over_fixed_rank64"],
        "KL lower": final["kl_gain_bootstrap_95ci"][0],
        "KL upper": final["kl_gain_bootstrap_95ci"][1],
        "first-token fidelity": final["first_token_fidelity"],
        "dense accuracy": final["generation"]["dense"]["accuracy"],
        "fixed rank-64 accuracy": final["generation"]["fixed_answer_fisher_r64"]["accuracy"],
        "routed accuracy": final["generation"]["cache_risk_rank_router"]["accuracy"],
        "storage ratio": final["budget"]["accounted_storage_ratio"],
    }]))
    display(pd.DataFrame([final["gate"]]))

## Takeaways

In [ ]:
print("Decision:", summary["decision"]["claim"])
print("External screen:", summary["decision"]["screen_passed"])
print("Locked final:", summary["decision"]["final_passed"])
for warning in summary["warnings"]:
    print("WARNING:", warning)

## 8. Save the complete audit trail

In [ ]:
output = pathlib.Path(OUTPUT_DIR)
assert (output / "summary.json").is_file()
assert (output / "cache_risk_rank_router.pt").is_file()
assert (output / "predictions.jsonl").is_file()
print("Kaggle output directory", output)
print("Publish this directory directly; no same-named archive is created.")